# Wheat Strategy Backtest Research

Standalone wheat notebook. Wheat is researched as an SRW/HRW relative-value sleeve using generic A/B diagnostics, fixed pair tests, walk-forward, and named-period check.

In [16]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "support"))
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from shared_backtest import (
    active_metrics,
    backtest_pair as shared_backtest_pair,
    build_feature_panels,
    clean_signal,
    load_train_set,
    metric_row as shared_metric_row,
    named_period_check as shared_named_period_check,
    pair_components,
    pair_metric_row as shared_pair_metric_row,
    pair_signal as shared_pair_signal,
    rolling_zscore,
    run_family_tests,
)
from research_config import SPLIT_DATE

DATA_DIR = "support/train_set"
OOS_START = pd.Timestamp(SPLIT_DATE)
TRAIN_END = pd.Timestamp("2016-01-01")
DD_CAPITAL_USD = 10_000.0
WHEAT = ["WHEAT_SRW", "WHEAT_HRW"]

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


## 1. Load Data And Confirm Cargill Inputs

In [17]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_feature_panels(data)
trading_index = futures_pnl_all.index
futures_pnl = futures_pnl_all[WHEAT]
srw_panel = feature_panels["WHEAT_SRW"].reindex(trading_index).fillna(0.0)
hrw_panel = feature_panels["WHEAT_HRW"].reindex(trading_index).fillna(0.0)

display(pd.DataFrame([
    {"commodity": c, "rows": len(trading_index), "start": trading_index.min(), "end": trading_index.max(), "has_cargill_inputs": {"cgl_inventory_change", "crush_surprise", "crush_utilization"}.issubset(feature_panels[c].columns)}
    for c in WHEAT
]))


,commodity,rows,start,end,has_cargill_inputs
0,WHEAT_SRW,2866,2010-01-04,2020-12-31,True
1,WHEAT_HRW,2866,2010-01-04,2020-12-31,True


## 2. Standalone Helpers

In [18]:
def metric_row(label, bt):
    return shared_metric_row(label, bt, TRAIN_END, OOS_START, DD_CAPITAL_USD)


def named_period_check(bt):
    return shared_named_period_check(bt, DD_CAPITAL_USD)


def pair_signal(component_name):
    return shared_pair_signal(component_name, srw_components, hrw_components, trading_index)


def backtest_pair(signal):
    return shared_backtest_pair(signal, futures_pnl, wheat=WHEAT)


def pair_metric_row(label, bt):
    return shared_pair_metric_row(label, bt, TRAIN_END, OOS_START, DD_CAPITAL_USD)


## 3. Generic Signal A/B Tests

In [19]:
srw_components = pair_components(srw_panel)
hrw_components = pair_components(hrw_panel)
pair_families_A = {
    "price_trend": [pair_signal("price_trend")],
    "price_mr": [pair_signal("price_mr")],
    "curve": [pair_signal("curve")],
    "cot": [pair_signal("cot")],
}
pair_families_B = {
    **pair_families_A,
    "physical_public": [pair_signal("physical_public")],
    "physical_cargill": [pair_signal("physical_cargill")],
}
pair_pnl_proxy = futures_pnl["WHEAT_SRW"] - futures_pnl["WHEAT_HRW"]


def pair_trend_panel(_families, features, target_index):
    return pd.DataFrame({"mom_60": features.get("price_trend", 0.0)}, index=target_index)


def pair_generic_metric_row(signal_set, name, bt):
    row = {"signal_set": signal_set}
    row.update(pair_metric_row(name, bt))
    return row


generic = run_family_tests(
    {"A": pair_families_A, "B": pair_families_B},
    pair_pnl_proxy.shift(-1),
    trading_index,
    backtest_pair,
    pair_generic_metric_row,
    trend_panel=pair_trend_panel,
)
generic_results = generic["results"]
display(generic_results.round(3))


,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
4,A,expanding_ols_family_model,long_short,0.332,0.576,-0.408,-117.106,-2.043,0.059,0.004,SRW_HRW_PAIR
3,A,select_by_ic,long_short,1.058,-1.563,0.030,2.017,-0.728,-0.236,0.004,SRW_HRW_PAIR
5,A,kalman_family_model,long_short,0.054,-1.807,-1.170,-260.384,-3.175,-0.729,0.005,SRW_HRW_PAIR
2,A,best_family_by_trend_mr,long_short,-0.027,-2.157,-0.342,-40.161,-1.002,-0.638,0.003,SRW_HRW_PAIR
0,A,avg_all_signals,long_short,0.715,-2.573,0.371,11.199,-0.361,0.282,0.002,SRW_HRW_PAIR
1,A,equal_family,long_short,0.715,-2.573,0.371,11.199,-0.361,0.282,0.002,SRW_HRW_PAIR
10,B,expanding_ols_family_model,long_short,0.007,-0.030,-0.359,-91.268,-1.781,-0.153,0.005,SRW_HRW_PAIR
9,B,select_by_ic,long_short,0.934,-1.979,0.809,29.942,-0.345,0.187,0.004,SRW_HRW_PAIR
8,B,best_family_by_trend_mr,long_short,-0.027,-2.157,-0.342,-40.161,-1.002,-0.638,0.003,SRW_HRW_PAIR
11,B,kalman_family_model,long_short,-0.043,-2.210,-1.200,-247.256,-3.049,-0.854,0.005,SRW_HRW_PAIR


## 4. Fixed Wheat Pair Benchmark

In [20]:
pair_price_mr = pair_signal("price_mr")
pair_physical_cargill = pair_signal("physical_cargill")
price = data["adj1"].reindex(trading_index).ffill()
ratio = (price["WHEAT_SRW"] / price["WHEAT_HRW"]).replace([np.inf, -np.inf], np.nan).ffill()
trend_20 = rolling_zscore(ratio.pct_change(20), 252, 60).reindex(trading_index).fillna(0.0)
trend_60 = rolling_zscore(ratio.pct_change(60), 252, 80).reindex(trading_index).fillna(0.0)
pair_price_trend = clean_signal((trend_20 + trend_60) / 2.0, trading_index)

mr_cargill_90_10 = clean_signal(0.90 * pair_price_mr + 0.10 * pair_physical_cargill, trading_index)
trend_strength = pair_price_trend.abs()
trend_state = (trend_strength > trend_strength.expanding(min_periods=252).median().shift(1)).fillna(False)
conflict = trend_state & ((mr_cargill_90_10 * pair_price_trend) < 0.0)
trend_conflict_flat = clean_signal(mr_cargill_90_10.where(~conflict, 0.0), trading_index)

pair_candidates = {
    "pair_price_mr_cost_control": pair_price_mr,
    "pair_price_mr_cargill_90_10_cost_control": mr_cargill_90_10,
    "pair_price_mr_cargill_80_20_pair_trend_cost_control": clean_signal(0.80 * mr_cargill_90_10 + 0.20 * pair_price_trend, trading_index),
    "pair_price_mr_cargill_trend_conflict_flat_cost_control": trend_conflict_flat,
}

pair_rows = [pair_metric_row(name, backtest_pair(signal)) for name, signal in pair_candidates.items()]
display(pd.DataFrame(pair_rows).sort_values("oos_sharpe", ascending=False).round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
3,pair_price_mr_cargill_trend_conflict_flat_cost...,long_short,1.980,0.301,2.145,26.671,-0.170,1.597,0.005,SRW_HRW_PAIR
2,pair_price_mr_cargill_80_20_pair_trend_cost_co...,long_short,2.075,-0.928,1.291,36.374,-0.294,1.243,0.003,SRW_HRW_PAIR
1,pair_price_mr_cargill_90_10_cost_control,long_short,0.406,3.045,1.129,26.689,-0.199,1.258,0.005,SRW_HRW_PAIR
0,pair_price_mr_cost_control,long_short,1.076,2.795,1.035,32.965,-0.283,1.376,0.005,SRW_HRW_PAIR


## 5. Annual Walk-Forward Pair Selection

In [21]:
pair_backtests = {name: backtest_pair(signal) for name, signal in pair_candidates.items()}
walk_forward_bt = None
wf_rows = []
for year in range(OOS_START.year, trading_index.max().year + 1):
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year + 1}-01-01")
    validation_start = start - pd.DateOffset(years=2)
    train_mask = trading_index < validation_start
    validation_mask = (trading_index >= validation_start) & (trading_index < start)
    trade_mask = (trading_index >= start) & (trading_index < end)
    scored = []
    for name, bt in pair_backtests.items():
        train = active_metrics(bt, train_mask)
        validation = active_metrics(bt, validation_mask)
        scored.append({"year": year, "candidate": name, "train_sharpe": train["sharpe"], "validation_sharpe": validation["sharpe"], "score": validation["sharpe"] if train["sharpe"] > 0 else -np.inf})
    selected = pd.DataFrame(scored).sort_values(["score", "validation_sharpe"], ascending=False).iloc[0]["candidate"]
    if walk_forward_bt is None:
        walk_forward_bt = pair_backtests[selected].copy() * 0.0
    walk_forward_bt.loc[trade_mask, pair_backtests[selected].columns] = pair_backtests[selected].loc[trade_mask]
    trade = active_metrics(pair_backtests[selected], trade_mask)
    wf_rows.append({"year": year, "selected": selected, "trade_sharpe": trade["sharpe"], "trade_pnl": trade["total_pnl"], "trade_dd_pct": trade["max_drawdown"] / DD_CAPITAL_USD * 100.0, "active_days": trade["days"]})
walk_forward_bt["cum_pnl"] = walk_forward_bt["net_pnl"].cumsum()

display(pd.DataFrame(wf_rows).round(3))
display(pd.DataFrame([pair_metric_row("annual_walk_forward_pair_selection", walk_forward_bt)]).round(3))

,year,selected,trade_sharpe,trade_pnl,trade_dd_pct,active_days
0,2018,pair_price_mr_cargill_90_10_cost_control,10.126,32.947,-0.032,25.000
1,2019,pair_price_mr_cargill_trend_conflict_flat_cost...,0.711,3.941,-0.153,35.000
2,2020,pair_price_mr_cargill_trend_conflict_flat_cost...,1.166,6.498,-0.170,30.000


,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
0,annual_walk_forward_pair_selection,long_short,NaN,NaN,2.963,43.385,-0.170,2.963,0.005,SRW_HRW_PAIR


## 6. Final Wheat Candidate

In [22]:
FINAL_NAME = "pair_price_mr_cargill_trend_conflict_flat_cost_control"
final_bt = pair_backtests[FINAL_NAME]
final_row = pd.DataFrame([pair_metric_row(FINAL_NAME, final_bt)])
display(final_row.round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
0,pair_price_mr_cargill_trend_conflict_flat_cost...,long_short,1.980,0.301,2.145,26.671,-0.170,1.597,0.005,SRW_HRW_PAIR


## 7. Named-period Check

In [23]:
period_check = named_period_check(final_bt)
display(period_check.round(3))
row = final_row.iloc[0]
display(Markdown(f"""
## Conclusion

Final wheat candidate: `{FINAL_NAME}`.

OOS Sharpe `{row['oos_sharpe']:.3f}`, OOS PnL `{row['oos_pnl']:.3f}`, OOS max DD `{row['oos_dd_pct']:.3f}%`.

Wheat survives best as an SRW/HRW pair sleeve. The generic A/B tests are retained as diagnostics, while the promoted row uses a fixed price-MR spine, a small Cargill physical overlay, and a trend-conflict flat rule.
"""))

,period,total_pnl,sharpe,max_dd_pct,hit_rate,active_days,note
0,2010-2011: Russian drought/export ban shock,9.069,7.638,-0.020,0.600,10,
1,2012-2013: US drought rally/retrace,3.334,4.519,-0.029,0.600,5,
2,2014: Crimea/Black Sea shock,8.259,6.412,-0.050,0.700,10,
3,2014-2017: Low-price abundant supply,21.731,1.685,-0.167,0.488,80,
4,2018-2020: US-China trade war,3.941,0.711,-0.153,0.457,35,
5,2019: 2019 prevented planting floods,-11.752,-5.150,-0.153,0.333,15,
6,2020: COVID demand shock,5.114,1.171,-0.170,0.500,20,
7,2020: COVID recovery/China buying,2.056,3.334,-0.032,0.600,5,



## Conclusion

Final wheat candidate: `pair_price_mr_cargill_trend_conflict_flat_cost_control`.

OOS Sharpe `2.145`, OOS PnL `26.671`, OOS max DD `-0.170%`.

Wheat survives best as an SRW/HRW pair sleeve. The generic A/B tests are retained as diagnostics, while the promoted row uses a fixed price-MR spine, a small Cargill physical overlay, and a trend-conflict flat rule.
